In [1]:
%pip install mlflow -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 787.0/787.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
!uv pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=03185440-d874-4728-bbf2-dc3f40ebf65b
To: /kaggle/working/dataset.zip
100%|████████████████████████████████████████| 356M/356M [00:04<00:00, 86.8MB/s]


In [3]:
import sys

sys.path.append('/kaggle/input/datasets/maksimbessolitsyn/')

In [4]:
%pip install mlflow -qq

Note: you may need to restart the kernel to use updated packages.


In [5]:
import logging
import warnings
import os

warnings.filterwarnings("ignore", category=UserWarning, module=r"torch(\.|$)")
warnings.filterwarnings("ignore", category=FutureWarning, module=r"torch(\.|$)")
logging.getLogger("torch").setLevel(logging.ERROR)
logging.getLogger("torch._dynamo").setLevel(logging.ERROR)




In [6]:
LOG_DIR = "./mlruns"


In [7]:
DATA_DIR = "."
PATH_INTERACTIONS = os.path.join(DATA_DIR, "interactions.parquet")
PATH_EMBEDDINGS = os.path.join(DATA_DIR, "embeddings.parquet")
PATH_ARTISTS = os.path.join(DATA_DIR, "artists.parquet")
SEED = 42

In [8]:
from sasrec import run_ddp_training, ExperimentConfig

In [9]:
fixed_experiment_parameters = ExperimentConfig(
    graph=ExperimentConfig.GraphConfig(
        n_layers=4,
        d_model=256,
        n_heads=4,
        dropout=0.0,
        log_q_correction=1.0,
        is_cosine_similarity=True,
    ),
    data=ExperimentConfig.DataConfig(
        vocab_size=157_162,
        max_seq_len=100,
        bos=0,
        path_interactions=PATH_INTERACTIONS,
        path_embeddings=PATH_EMBEDDINGS,
        path_artists=PATH_ARTISTS,
        core_min_interaction_per_user=5,
        test_interval_seconds=7 * 24 * 60 * 60,
        max_train_events_per_user=100,
    ),
    tau=None,
    training_dataset=None,
    test_dataset=ExperimentConfig.TestDatasetConfig(
        batch_size=32,
        device="cuda",
    ),
    optimizer=None,
    scheduler=ExperimentConfig.SchedulerConfig(
        class_name=None,
        json_args={},
    ),
    training=ExperimentConfig.TrainingConfig(
        num_epochs=15,
        grad_clip=1.0,
        eval_every=1,
        logging=True,
        log_dir=LOG_DIR,
        seed=SEED,
    ),
    evaluator=ExperimentConfig.EvaluatorConfig(
        topk=100,
    ),
)

In [10]:
from dataclasses import replace
import torch

tau = ExperimentConfig.TauConfig(
    class_name="CosPerUserTau",
    json_args={
        "initial_tau": 0.45,
        "tau_min": None,
        "tau_max": None,
        "num_epochs": 15,
        "num_tokens_per_epoch": 4_019_032,
    },
)

training_dataset = ExperimentConfig.TrainingDatasetConfig(
    batch_size=128,
    device="cuda",
    chunk_rows=64000,
    shuffle=True,
    seed=42,
    pin_memory=True,
    uniform_negative_items=None,
    in_batch_negative_items=None,
)

optimizer = ExperimentConfig.OptimizerConfig(
    class_name="AdamW",
     json_args={
        "lr": 3e-3,
        "weight_decay": 1e-5,
    },
)

for tau_min, tau_max in [(0.04, 0.05), (0.04, 0.055), (0.045, 0.055), (0.045, 0.06)]:
    print(f"Running experiment with tau_min={tau_min} and tau_max={tau_max}...")

    tau.json_args["tau_min"] = tau_min
    tau.json_args["tau_max"] = tau_max

    for uniform, unigram in [(18_000, 12_000), (22_000, 8_000), (26_000, 4_000)]:
        run_ddp_training(
            replace(
                fixed_experiment_parameters, 
                tau=tau,
                training_dataset=replace(
                    training_dataset,
                    uniform_negative_items=uniform, 
                    in_batch_negative_items=unigram,
                ),
                optimizer=optimizer
            ),
            world_size=torch.cuda.device_count()
        )


Running experiment with tau_min=0.04 and tau_max=0.05...


Epochs: 100%|██████████| 15/15 [35:08<00:00, 140.57s/it, train_loss=7.9097]



[Epoch 0] Train Loss: 11.6463 | Validation: hitrate: 0.1214, recall: 0.0327, ndcg: 0.0114, coverage: 0.0020

[Epoch 1] Train Loss: 10.8930 | Validation: hitrate: 0.1660, recall: 0.0463, ndcg: 0.0157, coverage: 0.0032

[Epoch 2] Train Loss: 10.6276 | Validation: hitrate: 0.2199, recall: 0.0661, ndcg: 0.0261, coverage: 0.0107

[Epoch 3] Train Loss: 10.2807 | Validation: hitrate: 0.2379, recall: 0.0718, ndcg: 0.0270, coverage: 0.0256

[Epoch 4] Train Loss: 10.0194 | Validation: hitrate: 0.2658, recall: 0.0811, ndcg: 0.0311, coverage: 0.0521

[Epoch 5] Train Loss: 9.7511 | Validation: hitrate: 0.2997, recall: 0.0943, ndcg: 0.0366, coverage: 0.1024

[Epoch 6] Train Loss: 9.4871 | Validation: hitrate: 0.3234, recall: 0.1054, ndcg: 0.0408, coverage: 0.1515

[Epoch 7] Train Loss: 9.2584 | Validation: hitrate: 0.3468, recall: 0.1171, ndcg: 0.0465, coverage: 0.2192

[Epoch 8] Train Loss: 9.0657 | Validation: hitrate: 0.3475, recall: 0.1175, ndcg: 0.0470, coverage: 0.2957

[Epoch 9] Train Loss: 

Epochs: 100%|██████████| 15/15 [35:25<00:00, 141.73s/it, train_loss=7.8389]



[Epoch 0] Train Loss: 11.3248 | Validation: hitrate: 0.1202, recall: 0.0323, ndcg: 0.0106, coverage: 0.0016

[Epoch 1] Train Loss: 10.5841 | Validation: hitrate: 0.1409, recall: 0.0380, ndcg: 0.0134, coverage: 0.0032

[Epoch 2] Train Loss: 10.3466 | Validation: hitrate: 0.2041, recall: 0.0605, ndcg: 0.0226, coverage: 0.0080

[Epoch 3] Train Loss: 10.0459 | Validation: hitrate: 0.2386, recall: 0.0722, ndcg: 0.0275, coverage: 0.0205

[Epoch 4] Train Loss: 9.7811 | Validation: hitrate: 0.2538, recall: 0.0767, ndcg: 0.0293, coverage: 0.0380

[Epoch 5] Train Loss: 9.5290 | Validation: hitrate: 0.2896, recall: 0.0901, ndcg: 0.0345, coverage: 0.0732

[Epoch 6] Train Loss: 9.2668 | Validation: hitrate: 0.3208, recall: 0.1043, ndcg: 0.0409, coverage: 0.1260

[Epoch 7] Train Loss: 9.0254 | Validation: hitrate: 0.3328, recall: 0.1092, ndcg: 0.0428, coverage: 0.1877

[Epoch 8] Train Loss: 8.8324 | Validation: hitrate: 0.3451, recall: 0.1149, ndcg: 0.0461, coverage: 0.2520

[Epoch 9] Train Loss: 8

...


[Epoch 0] Train Loss: 11.6221 | Validation: hitrate: 0.1205, recall: 0.0329, ndcg: 0.0117, coverage: 0.0015

[Epoch 1] Train Loss: 10.9685 | Validation: hitrate: 0.1550, recall: 0.0424, ndcg: 0.0152, coverage: 0.0029

[Epoch 2] Train Loss: 10.7626 | Validation: hitrate: 0.2076, recall: 0.0624, ndcg: 0.0231, coverage: 0.0068

[Epoch 3] Train Loss: 10.4884 | Validation: hitrate: 0.2360, recall: 0.0711, ndcg: 0.0268, coverage: 0.0194

[Epoch 4] Train Loss: 10.1848 | Validation: hitrate: 0.2585, recall: 0.0791, ndcg: 0.0300, coverage: 0.0477

[Epoch 5] Train Loss: 9.9095 | Validation: hitrate: 0.2916, recall: 0.0917, ndcg: 0.0353, coverage: 0.0810

[Epoch 6] Train Loss: 9.6356 | Validation: hitrate: 0.3229, recall: 0.1050, ndcg: 0.0410, coverage: 0.1447

[Epoch 7] Train Loss: 9.3988 | Validation: hitrate: 0.3390, recall: 0.1132, ndcg: 0.0445, coverage: 0.1997

[Epoch 8] Train Loss: 9.2014 | Validation: hitrate: 0.3472, recall: 0.1171, ndcg: 0.0469, coverage: 0.2766

[Epoch 9] Train Loss: 

Epochs: 100%|██████████| 15/15 [35:28<00:00, 141.91s/it, train_loss=8.3734]



[Epoch 0] Train Loss: 11.2477 | Validation: hitrate: 0.1203, recall: 0.0335, ndcg: 0.0113, coverage: 0.0009

[Epoch 1] Train Loss: 10.7160 | Validation: hitrate: 0.1521, recall: 0.0405, ndcg: 0.0146, coverage: 0.0020

[Epoch 2] Train Loss: 10.5328 | Validation: hitrate: 0.1580, recall: 0.0434, ndcg: 0.0154, coverage: 0.0030

[Epoch 3] Train Loss: 10.4089 | Validation: hitrate: 0.1670, recall: 0.0467, ndcg: 0.0170, coverage: 0.0061

[Epoch 4] Train Loss: 10.2497 | Validation: hitrate: 0.2389, recall: 0.0734, ndcg: 0.0282, coverage: 0.0120

[Epoch 5] Train Loss: 10.0290 | Validation: hitrate: 0.2541, recall: 0.0793, ndcg: 0.0309, coverage: 0.0208

[Epoch 6] Train Loss: 9.8409 | Validation: hitrate: 0.2677, recall: 0.0826, ndcg: 0.0326, coverage: 0.0428

[Epoch 7] Train Loss: 9.6608 | Validation: hitrate: 0.2881, recall: 0.0917, ndcg: 0.0360, coverage: 0.0656

[Epoch 8] Train Loss: 9.4368 | Validation: hitrate: 0.3107, recall: 0.1011, ndcg: 0.0402, coverage: 0.1076

[Epoch 9] Train Loss:

Epochs: 100%|██████████| 15/15 [35:31<00:00, 142.11s/it, train_loss=7.3651]



[Epoch 0] Train Loss: 10.6816 | Validation: hitrate: 0.1336, recall: 0.0375, ndcg: 0.0122, coverage: 0.0013

[Epoch 1] Train Loss: 10.0717 | Validation: hitrate: 0.1475, recall: 0.0395, ndcg: 0.0143, coverage: 0.0030

[Epoch 2] Train Loss: 9.8855 | Validation: hitrate: 0.1900, recall: 0.0552, ndcg: 0.0201, coverage: 0.0065

[Epoch 3] Train Loss: 9.6230 | Validation: hitrate: 0.2282, recall: 0.0690, ndcg: 0.0265, coverage: 0.0191

[Epoch 4] Train Loss: 9.3253 | Validation: hitrate: 0.2640, recall: 0.0821, ndcg: 0.0317, coverage: 0.0409

[Epoch 5] Train Loss: 9.0522 | Validation: hitrate: 0.2862, recall: 0.0899, ndcg: 0.0346, coverage: 0.0823

[Epoch 6] Train Loss: 8.7804 | Validation: hitrate: 0.3208, recall: 0.1043, ndcg: 0.0412, coverage: 0.1315

[Epoch 7] Train Loss: 8.5406 | Validation: hitrate: 0.3364, recall: 0.1120, ndcg: 0.0441, coverage: 0.2115

[Epoch 8] Train Loss: 8.3469 | Validation: hitrate: 0.3441, recall: 0.1149, ndcg: 0.0460, coverage: 0.2608

[Epoch 9] Train Loss: 8.1